In [ ]:

import os
import platform
import time
import uuid
import json
import subprocess
import importlib.metadata
import datetime
import shutil

RUN_ID = str(uuid.uuid4())
TIMESTAMP = datetime.datetime.now(datetime.timezone.utc).isoformat()
PROBE_VERSION = "1.1.0"
WORKING_DIR = "/kaggle/working/"
SCHEMA_VERSION = "1.1"

# Prepare data structures
runtime_data = {
    "schema_version": SCHEMA_VERSION,
    "probe_version": PROBE_VERSION,
    "run_id": RUN_ID,
    "utc_timestamp": TIMESTAMP,
    "python_version": platform.python_version(),
    "python_executable": os.sys.executable,
    "platform": platform.platform(),
    "architecture": platform.machine(),
    "processor": platform.processor(),
    "hostname": platform.node(),
    "cpu_count": os.cpu_count(),
    "total_disk_space": shutil.disk_usage('/').total,
    "available_disk_space": shutil.disk_usage('/').free,
    "current_working_directory": os.getcwd()
}

# Optional psutil
try:
    import psutil
    runtime_data["total_system_memory"] = psutil.virtual_memory().total
    runtime_data["available_system_memory"] = psutil.virtual_memory().available
except Exception as e:
    runtime_data["psutil_memory_error"] = str(e)
    # fallback to meminfo
    try:
        with open("/proc/meminfo", "r") as f:
            runtime_data["proc_meminfo"] = f.read(1024)
    except Exception:
        pass

gpu_data = {}
inventory = {
    "schema_version": SCHEMA_VERSION,
    "probe_version": PROBE_VERSION,
    "run_id": RUN_ID,
    "utc_timestamp": TIMESTAMP,
    "required_packages": {},
    "full_inventory": {}
}
smoke_results = {
    "schema_version": SCHEMA_VERSION,
    "probe_version": PROBE_VERSION,
    "run_id": RUN_ID,
    "utc_timestamp": TIMESTAMP,
    "checks": {}
}
summary = {
    "schema_version": SCHEMA_VERSION,
    "probe_version": PROBE_VERSION,
    "run_id": RUN_ID,
    "utc_timestamp": TIMESTAMP,
    "statuses": {}
}

pip_freeze_output = ""
nvidia_smi_log = ""
execution_status = "ERROR"

try:
    # --- NVIDIA SMI ---
    try:
        res = subprocess.run(["nvidia-smi"], capture_output=True, text=True, check=False)
        gpu_data["nvidia_smi"] = {
            "command": "nvidia-smi",
            "return_code": res.returncode,
            "stdout": res.stdout,
            "stderr": res.stderr,
            "success": res.returncode == 0
        }
        nvidia_smi_log = res.stdout + "\n" + res.stderr
    except Exception as e:
        gpu_data["nvidia_smi"] = {
            "command": "nvidia-smi",
            "return_code": -1,
            "stdout": "",
            "stderr": str(e),
            "success": False
        }
        nvidia_smi_log = str(e)

    # --- TORCH GPU INFO ---
    try:
        import torch
        gpu_data["cuda_available"] = torch.cuda.is_available()
        gpu_data["cuda_device_count"] = torch.cuda.device_count()
        gpu_data["devices"] = []
        for i in range(torch.cuda.device_count()):
            props = torch.cuda.get_device_properties(i)
            dev_info = {
                "id": i,
                "name": torch.cuda.get_device_name(i),
                "compute_capability": torch.cuda.get_device_capability(i),
                "total_memory": props.total_memory,
                "allocated_memory": torch.cuda.memory_allocated(i),
                "reserved_memory": torch.cuda.memory_reserved(i)
            }
            gpu_data["devices"].append(dev_info)
        gpu_data["pytorch_version"] = torch.__version__
        gpu_data["pytorch_cuda_build_version"] = torch.version.cuda
        gpu_data["cudnn_version"] = torch.backends.cudnn.version()
        gpu_data["is_bf16_supported"] = torch.cuda.is_bf16_supported()
        gpu_data["tf32_available"] = torch.backends.cuda.matmul.allow_tf32
        gpu_data["sdpa_exists"] = hasattr(torch.nn.functional, 'scaled_dot_product_attention')
    except Exception as e:
        gpu_data["torch_error"] = str(e)
    
    runtime_data["gpu_observations"] = gpu_data

    # --- PACKAGE INVENTORY ---
    packages_map = {
        "torch": "torch",
        "torchvision": "torchvision",
        "torchaudio": "torchaudio",
        "triton": "triton",
        "transformers": "transformers",
        "datasets": "datasets",
        "accelerate": "accelerate",
        "peft": "peft",
        "trl": "trl",
        "huggingface-hub": "huggingface_hub",
        "bitsandbytes": "bitsandbytes",
        "captum": "captum",
        "numpy": "numpy",
        "scipy": "scipy",
        "pandas": "pandas",
        "lm-eval": "lm_eval",
        "hydra-core": "hydra",
        "omegaconf": "omegaconf",
        "matplotlib": "matplotlib",
        "seaborn": "seaborn",
        "typer": "typer",
        "rich": "rich",
        "tqdm": "tqdm",
        "jsonlines": "jsonlines",
        "pyyaml": "yaml",
        "pytest": "pytest",
        "ruff": "ruff"
    }

    import importlib
    for dist_name, mod_name in packages_map.items():
        pkg_info = {
            "distribution": dist_name,
            "module": mod_name,
            "installed": False,
            "version": None,
            "distribution_error": None,
            "import_success": False,
            "import_error": None
        }
        try:
            pkg_info["version"] = importlib.metadata.version(dist_name)
            pkg_info["installed"] = True
        except importlib.metadata.PackageNotFoundError:
            pkg_info["distribution_error"] = "PackageNotFoundError"
        except Exception as e:
            pkg_info["distribution_error"] = type(e).__name__ + ": " + str(e)
        
        if pkg_info["installed"]:
            try:
                importlib.import_module(mod_name)
                pkg_info["import_success"] = True
            except Exception as e:
                pkg_info["import_error"] = type(e).__name__ + ": " + str(e)
        
        inventory["required_packages"][dist_name] = pkg_info

    try:
        all_dists = list(importlib.metadata.distributions())
        for dist in sorted(all_dists, key=lambda d: (d.metadata['Name'] or '').lower()):
            name = dist.metadata['Name']
            if name:
                inventory["full_inventory"][name.lower()] = dist.version
    except Exception as e:
        inventory["full_inventory"]["error"] = str(e)

    try:
        res = subprocess.run([os.sys.executable, "-m", "pip", "freeze"], capture_output=True, text=True, check=False)
        pip_freeze_output = res.stdout + "\n" + res.stderr
    except Exception as e:
        pip_freeze_output = str(e)

    # --- SMOKE CHECKS ---
    def run_check(name, fn):
        try:
            obs = fn()
            smoke_results["checks"][name] = {
                "status": "PASS",
                "observed": obs if obs else {},
                "exception_type": None,
                "exception_message": None
            }
        except Exception as e:
            smoke_results["checks"][name] = {
                "status": "FAIL",
                "observed": {},
                "exception_type": type(e).__name__,
                "exception_message": str(e)
            }

    def cpu_tensor_check():
        import torch
        t = torch.ones(5)
        return {"sum": (t + t).sum().item()}
    run_check("cpu_tensor", cpu_tensor_check)

    def cuda_fp16_creation():
        import torch
        if not torch.cuda.is_available(): raise RuntimeError("CUDA not available")
        t = torch.ones(5, device="cuda:0", dtype=torch.float16)
        return {"device": str(t.device), "dtype": str(t.dtype)}
    run_check("cuda_fp16_creation", cuda_fp16_creation)

    def cuda_fp16_matmul():
        import torch
        if not torch.cuda.is_available(): raise RuntimeError("CUDA not available")
        a = torch.randn(10, 10, device="cuda:0", dtype=torch.float16)
        b = torch.randn(10, 10, device="cuda:0", dtype=torch.float16)
        c = torch.matmul(a, b)
        return {"shape": list(c.shape)}
    run_check("cuda_fp16_matmul", cuda_fp16_matmul)

    def sdpa_check():
        import torch
        import torch.nn.functional as F
        if not torch.cuda.is_available(): raise RuntimeError("CUDA not available")
        q = torch.randn(1, 1, 10, 10, device="cuda:0", dtype=torch.float16)
        k = torch.randn(1, 1, 10, 10, device="cuda:0", dtype=torch.float16)
        v = torch.randn(1, 1, 10, 10, device="cuda:0", dtype=torch.float16)
        out = F.scaled_dot_product_attention(q, k, v)
        return {"shape": list(out.shape)}
    run_check("sdpa", sdpa_check)

    def nf4_check():
        import torch
        if not inventory["required_packages"].get("bitsandbytes", {}).get("import_success"):
            raise NotImplementedError("bitsandbytes not importable")
        if not torch.cuda.is_available(): raise RuntimeError("CUDA not available")
        import bitsandbytes as bnb
        t = torch.randn(16, 16, device="cuda:0", dtype=torch.float16)
        out, state = bnb.functional.quantize_4bit(t, quant_type="nf4")
        deq = bnb.functional.dequantize_4bit(out, state, quant_type="nf4")
        return {"quantized_shape": list(out.shape), "dequantized_shape": list(deq.shape)}
    
    try:
        nf4_res = nf4_check()
        smoke_results["checks"]["nf4_quantise"] = {
            "status": "PASS", "observed": nf4_res, "exception_type": None, "exception_message": None
        }
    except NotImplementedError as e:
        smoke_results["checks"]["nf4_quantise"] = {
            "status": "UNAVAILABLE", "observed": {}, "exception_type": type(e).__name__, "exception_message": str(e)
        }
    except Exception as e:
        smoke_results["checks"]["nf4_quantise"] = {
            "status": "FAIL", "observed": {}, "exception_type": type(e).__name__, "exception_message": str(e)
        }

    try:
        import torch
        peak = torch.cuda.max_memory_allocated(0) if torch.cuda.is_available() else 0
        smoke_results["checks"]["peak_memory"] = {
            "status": "PASS", "observed": {"peak_allocated_bytes": peak}, "exception_type": None, "exception_message": None
        }
    except Exception:
        pass

    try:
        import torch
        if torch.cuda.is_available(): torch.cuda.empty_cache()
    except Exception:
        pass

    execution_status = "PASS"

except Exception as e:
    execution_status = "ERROR"
    summary["execution_error"] = str(e)
finally:
    # --- STATUS CALCULATION ---
    t4_hardware_status = "FAIL"
    try:
        for dev in gpu_data.get("devices", []):
            if "T4" in dev.get("name", ""):
                t4_hardware_status = "PASS"
                break
    except Exception:
        pass

    cuda_smoke_status = "FAIL"
    try:
        c_create = smoke_results["checks"].get("cuda_fp16_creation", {}).get("status") == "PASS"
        c_matmul = smoke_results["checks"].get("cuda_fp16_matmul", {}).get("status") == "PASS"
        if c_create and c_matmul:
            cuda_smoke_status = "PASS"
    except Exception:
        pass

    sdpa_smoke_status = "FAIL"
    try:
        if smoke_results["checks"].get("sdpa", {}).get("status") == "PASS":
            sdpa_smoke_status = "PASS"
    except Exception:
        pass

    nf4_smoke_status = smoke_results["checks"].get("nf4_quantise", {}).get("status", "FAIL")

    hardware_gate_status = "FAIL"
    if t4_hardware_status == "PASS" and cuda_smoke_status == "PASS" and sdpa_smoke_status == "PASS":
        hardware_gate_status = "PASS"

    dependency_readiness = "NOT_READY"
    fundamental_pkgs = ["torch", "transformers", "datasets", "accelerate", "numpy"]
    missing_fundamental = [p for p in fundamental_pkgs if not inventory.get("required_packages", {}).get(p, {}).get("import_success")]
    if not missing_fundamental:
        all_required = True
        for pkg, info in inventory.get("required_packages", {}).items():
            # Exclude dev tools from readiness
            if pkg in ["pytest", "ruff", "tqdm", "jsonlines", "pyyaml", "typer", "rich", "matplotlib", "seaborn", "lm-eval", "hydra-core", "omegaconf", "peft", "trl", "bitsandbytes", "captum"]:
                continue
            if not info.get("import_success"):
                all_required = False
                break
        dependency_readiness = "READY" if all_required else "PARTIAL"

    if execution_status == "ERROR":
        overall_gate0_status = "ERROR"
    elif hardware_gate_status == "PASS" and nf4_smoke_status == "PASS" and dependency_readiness == "READY":
        overall_gate0_status = "PASS"
    elif hardware_gate_status == "PASS":
        overall_gate0_status = "PARTIAL"
    else:
        overall_gate0_status = "FAIL"

    summary["statuses"] = {
        "execution_status": execution_status,
        "hardware_gate_status": hardware_gate_status,
        "nf4_status": nf4_smoke_status,
        "dependency_readiness": dependency_readiness,
        "overall_gate0_status": overall_gate0_status
    }

    # --- WRITE ARTIFACTS ---
    os.makedirs(WORKING_DIR, exist_ok=True)

    with open(os.path.join(WORKING_DIR, "gate0_environment_manifest.json"), "w") as f:
        json.dump(runtime_data, f, indent=2)

    with open(os.path.join(WORKING_DIR, "gate0_package_inventory.json"), "w") as f:
        json.dump(inventory, f, indent=2)

    with open(os.path.join(WORKING_DIR, "pip_freeze.log"), "w") as f:
        f.write(pip_freeze_output)

    with open(os.path.join(WORKING_DIR, "nvidia_smi.log"), "w") as f:
        f.write(nvidia_smi_log)

    with open(os.path.join(WORKING_DIR, "gate0_smoke_tests.json"), "w") as f:
        json.dump(smoke_results, f, indent=2)

    with open(os.path.join(WORKING_DIR, "gate0_summary.json"), "w") as f:
        json.dump(summary, f, indent=2)

    print(json.dumps(summary, indent=2))
